# 📦 Notebook 00 — Setup & Organize
## Two-Phase: Drive Organization → HuggingFace Upload

---
## 🗺️ How to use this notebook:

| Phase | What it does | When to run |
|---|---|---|
| **PHASE 1** | Copies all project files into one organized folder in Drive | Run now |
| **PHASE 2** | Pushes that organized folder to HuggingFace Hub | Run AFTER you verify Phase 1 |

---
### Organized folder structure (created in Drive):
```
MyDrive/
└── Quantum_LPR_Export/          ← new organized folder
    ├── dataset/
    │   ├── wYe7pBJ7-train.zip   ← your training images
    │   └── 2_train_hr_images.csv
    ├── checkpoints/
    │   ├── quantum/
    │   │   ├── latest.pth       ← most recent epoch
    │   │   ├── best.pth         ← best by val CER
    │   │   └── history.json     ← training curve data
    │   └── classical/
    │       ├── latest.pth
    │       ├── best.pth
    │       └── history.json
    └── meta/
        └── test_indices.json
```

---
# ⚙️ Cell 0 — Common Setup (run this first always)

In [1]:
import os, shutil, json
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────
# 🔧 YOUR SOURCE PATHS (where your files currently live)
# ─────────────────────────────────────────────────────────────
ZIP_SRC          = '/content/drive/MyDrive/License Plate/wYe7pBJ7-train.zip'
CSV_SRC          = '/content/drive/MyDrive/License Plate/Manifests/2_train_hr_images.csv'
PROJECT_SRC      = '/content/drive/MyDrive/Quantum_LPR_Project'
CHECKPOINT_SRC   = os.path.join(PROJECT_SRC, 'checkpoints')
HISTORY_SRC      = os.path.join(PROJECT_SRC, 'history')
TEST_IDX_SRC     = os.path.join(PROJECT_SRC, 'test_indices.json')

# ─────────────────────────────────────────────────────────────
# 🎯 EXPORT FOLDER (organized, ready for HF)
# ─────────────────────────────────────────────────────────────
EXPORT_ROOT = '/content/drive/MyDrive/Quantum_LPR_Export'

# Subfolder paths
EXPORT_DATASET    = os.path.join(EXPORT_ROOT, 'dataset')
EXPORT_Q_CKPT     = os.path.join(EXPORT_ROOT, 'checkpoints', 'quantum')
EXPORT_C_CKPT     = os.path.join(EXPORT_ROOT, 'checkpoints', 'classical')
EXPORT_META       = os.path.join(EXPORT_ROOT, 'meta')

for d in [EXPORT_DATASET, EXPORT_Q_CKPT, EXPORT_C_CKPT, EXPORT_META]:
    os.makedirs(d, exist_ok=True)

print('✅ Paths configured.')
print(f'\n  Export folder: {EXPORT_ROOT}')
print(f'\n  Checking sources...')

sources = {
    'Training ZIP'     : ZIP_SRC,
    'CSV Manifest'     : CSV_SRC,
    'Checkpoint dir'   : CHECKPOINT_SRC,
    'History dir'      : HISTORY_SRC,
    'test_indices.json': TEST_IDX_SRC,
}
for name, path in sources.items():
    exists = os.path.exists(path)
    if exists and os.path.isfile(path):
        size = f'{os.path.getsize(path)/1e6:.1f} MB'
    elif exists and os.path.isdir(path):
        files = os.listdir(path)
        size  = f'{len(files)} files inside'
    else:
        size = 'NOT FOUND'
    print(f'  {"✅" if exists else "❌"}  {name}: {size}')

Mounted at /content/drive
✅ Paths configured.

  Export folder: /content/drive/MyDrive/Quantum_LPR_Export

  Checking sources...
  ✅  Training ZIP: 719.7 MB
  ✅  CSV Manifest: 9.1 MB
  ✅  Checkpoint dir: 3 files inside
  ✅  History dir: 1 files inside
  ✅  test_indices.json: 0.1 MB


---
# 🗂️ PHASE 1 — Organize Files into Drive Export Folder
---

## Phase 1 — Cell A: Copy Dataset Files

In [2]:
def copy_file(src, dst, label):
    """Copy a file with progress info. Skips if dst already exists."""
    if os.path.exists(dst):
        size = os.path.getsize(dst) / 1e6
        print(f'  ✅ Already exists: {label} ({size:.1f} MB) — skipping')
        return True
    if not os.path.exists(src):
        print(f'  ❌ Source not found: {label} → {src}')
        return False
    print(f'  📋 Copying {label} ...')
    shutil.copy2(src, dst)
    size = os.path.getsize(dst) / 1e6
    print(f'  ✅ Done: {label} ({size:.1f} MB)')
    return True

print('📂 PHASE 1A — Copying dataset files...')
print('─' * 50)

r1 = copy_file(ZIP_SRC, os.path.join(EXPORT_DATASET, 'wYe7pBJ7-train.zip'), 'Training ZIP')
r2 = copy_file(CSV_SRC, os.path.join(EXPORT_DATASET, '2_train_hr_images.csv'), 'CSV Manifest')

print(f'\n  Summary:')
print(f'  Training ZIP  → {EXPORT_DATASET}/wYe7pBJ7-train.zip   {"✅" if r1 else "❌"}')
print(f'  CSV Manifest  → {EXPORT_DATASET}/2_train_hr_images.csv  {"✅" if r2 else "❌"}')

📂 PHASE 1A — Copying dataset files...
──────────────────────────────────────────────────
  📋 Copying Training ZIP ...
  ✅ Done: Training ZIP (719.7 MB)
  📋 Copying CSV Manifest ...
  ✅ Done: CSV Manifest (9.1 MB)

  Summary:
  Training ZIP  → /content/drive/MyDrive/Quantum_LPR_Export/dataset/wYe7pBJ7-train.zip   ✅
  CSV Manifest  → /content/drive/MyDrive/Quantum_LPR_Export/dataset/2_train_hr_images.csv  ✅


## Phase 1 — Cell B: Copy Quantum Checkpoints

In [3]:
print('📂 PHASE 1B — Copying QUANTUM checkpoints...')
print('─' * 50)

# Map of: (source filename, destination filename, label)
quantum_files = [
    ('8qubit_model.pth',  'latest.pth',  'Quantum Latest Checkpoint'),
    ('8qubit_best.pth',   'best.pth',    'Quantum Best Checkpoint'),
]

for src_name, dst_name, label in quantum_files:
    src = os.path.join(CHECKPOINT_SRC, src_name)
    dst = os.path.join(EXPORT_Q_CKPT, dst_name)

    if not os.path.exists(src):
        # If 8qubit_best.pth doesn't exist, use 8qubit_model.pth as both
        if src_name == '8qubit_best.pth':
            alt_src = os.path.join(CHECKPOINT_SRC, '8qubit_model.pth')
            if os.path.exists(alt_src):
                print(f'  ℹ️  best.pth not found — using latest as best')
                copy_file(alt_src, dst, label)
            else:
                print(f'  ❌ Neither best nor latest checkpoint found!')
        else:
            print(f'  ❌ Not found: {src}')
    else:
        copy_file(src, dst, label)

# Copy training history JSON
q_hist_src = os.path.join(HISTORY_SRC, 'Quantum_history.json')
q_hist_dst = os.path.join(EXPORT_Q_CKPT, 'history.json')
if os.path.exists(q_hist_src):
    copy_file(q_hist_src, q_hist_dst, 'Quantum History JSON')
else:
    # Create an empty history if none exists
    empty = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_wer': []}
    with open(q_hist_dst, 'w') as f:
        json.dump(empty, f)
    print(f'  ℹ️  No history found — created empty history.json')

# List what we have
print(f'\n  Contents of {EXPORT_Q_CKPT}:')
for f in sorted(os.listdir(EXPORT_Q_CKPT)):
    size = os.path.getsize(os.path.join(EXPORT_Q_CKPT, f)) / 1e6
    print(f'    📄 {f}  ({size:.2f} MB)')

📂 PHASE 1B — Copying QUANTUM checkpoints...
──────────────────────────────────────────────────
  📋 Copying Quantum Latest Checkpoint ...
  ✅ Done: Quantum Latest Checkpoint (2.8 MB)
  📋 Copying Quantum Best Checkpoint ...
  ✅ Done: Quantum Best Checkpoint (2.8 MB)
  📋 Copying Quantum History JSON ...
  ✅ Done: Quantum History JSON (0.0 MB)

  Contents of /content/drive/MyDrive/Quantum_LPR_Export/checkpoints/quantum:
    📄 best.pth  (2.83 MB)
    📄 history.json  (0.00 MB)
    📄 latest.pth  (2.83 MB)


## Phase 1 — Cell C: Copy Classical Checkpoints

In [4]:
print('📂 PHASE 1C — Copying CLASSICAL checkpoints...')
print('─' * 50)

classical_files = [
    ('classical_model.pth', 'latest.pth', 'Classical Latest Checkpoint'),
    ('classical_best.pth',  'best.pth',   'Classical Best Checkpoint'),
]

c_any = False
for src_name, dst_name, label in classical_files:
    src = os.path.join(CHECKPOINT_SRC, src_name)
    dst = os.path.join(EXPORT_C_CKPT, dst_name)
    if os.path.exists(src):
        copy_file(src, dst, label)
        c_any = True
    else:
        print(f'  ℹ️  {label} not found — will create fresh in Kaggle training')

if not c_any:
    print('  ℹ️  No classical checkpoints yet — that is OK!')
    print('       Classical model will train from scratch on Kaggle.')

# Classical history
c_hist_src = os.path.join(HISTORY_SRC, 'Classical_history.json')
c_hist_dst = os.path.join(EXPORT_C_CKPT, 'history.json')
if os.path.exists(c_hist_src):
    copy_file(c_hist_src, c_hist_dst, 'Classical History JSON')
else:
    empty = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_wer': []}
    with open(c_hist_dst, 'w') as f:
        json.dump(empty, f)
    print(f'  ℹ️  Created empty classical history.json')

print(f'\n  Contents of {EXPORT_C_CKPT}:')
for f in sorted(os.listdir(EXPORT_C_CKPT)):
    size = os.path.getsize(os.path.join(EXPORT_C_CKPT, f)) / 1e6
    print(f'    📄 {f}  ({size:.2f} MB)')

📂 PHASE 1C — Copying CLASSICAL checkpoints...
──────────────────────────────────────────────────
  ℹ️  Classical Latest Checkpoint not found — will create fresh in Kaggle training
  ℹ️  Classical Best Checkpoint not found — will create fresh in Kaggle training
  ℹ️  No classical checkpoints yet — that is OK!
       Classical model will train from scratch on Kaggle.
  ℹ️  Created empty classical history.json

  Contents of /content/drive/MyDrive/Quantum_LPR_Export/checkpoints/classical:
    📄 history.json  (0.00 MB)


## Phase 1 — Cell D: Copy Meta Files

In [5]:
print('📂 PHASE 1D — Copying meta files...')
print('─' * 50)

dst_idx = os.path.join(EXPORT_META, 'test_indices.json')
if os.path.exists(TEST_IDX_SRC):
    copy_file(TEST_IDX_SRC, dst_idx, 'test_indices.json')
else:
    print('  ℹ️  test_indices.json not found — will be created on first Kaggle run')

print(f'\n  Contents of {EXPORT_META}:')
for f in sorted(os.listdir(EXPORT_META)):
    size = os.path.getsize(os.path.join(EXPORT_META, f)) / 1e6
    print(f'    📄 {f}  ({size:.4f} MB)')

📂 PHASE 1D — Copying meta files...
──────────────────────────────────────────────────
  📋 Copying test_indices.json ...
  ✅ Done: test_indices.json (0.1 MB)

  Contents of /content/drive/MyDrive/Quantum_LPR_Export/meta:
    📄 test_indices.json  (0.1033 MB)


## Phase 1 — Cell E: ✅ Final Verification — Full Folder Summary

In [6]:
print('='*60)
print('  📋 PHASE 1 COMPLETE — Export Folder Contents')
print('='*60)
print(f'  Location: {EXPORT_ROOT}')
print()

total_size = 0
all_ok = True

for root, dirs, files in os.walk(EXPORT_ROOT):
    # only show relative path
    rel = root.replace(EXPORT_ROOT, '').lstrip('/')
    indent = '  ' * rel.count('/')
    if rel:
        print(f'{indent}📁 {os.path.basename(root)}/')
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        fsize = os.path.getsize(fpath)
        total_size += fsize
        fsize_str  = f'{fsize/1e6:.2f} MB' if fsize > 1000 else f'{fsize} B'
        fi = '  ' * (rel.count('/') + 1)
        print(f'{fi}📄 {fname}  ({fsize_str})')

print()
print(f'  Total size: {total_size/1e6:.1f} MB')
print('='*60)
print()

# Check critical files
critical = [
    ('dataset/wYe7pBJ7-train.zip',           '⚠️  Dataset ZIP missing!'),
    ('dataset/2_train_hr_images.csv',         '⚠️  CSV labels missing!'),
    ('checkpoints/quantum/latest.pth',        '❌  Quantum checkpoint MISSING — training cannot resume!'),
    ('checkpoints/quantum/history.json',      '⚠️  Quantum history missing'),
    ('checkpoints/classical/history.json',    '⚠️  Classical history missing'),
]

print('  Critical file check:')
for rel_path, warn in critical:
    full = os.path.join(EXPORT_ROOT, rel_path)
    if os.path.exists(full):
        size = os.path.getsize(full) / 1e6
        print(f'  ✅  {rel_path}  ({size:.2f} MB)')
    else:
        print(f'  ❌  {rel_path}  ← {warn}')
        if 'MISSING' in warn:
            all_ok = False

print()
if all_ok:
    print('  🎉 Phase 1 complete!')
    print('  👉 Open your Drive and browse to: MyDrive/Quantum_LPR_Export/')
    print('  👉 Verify the files look correct')
    print('  👉 Then run PHASE 2 cells to push to HuggingFace')
else:
    print('  ⚠️  Some critical files are missing — check the paths in Cell 0 before proceeding')

  📋 PHASE 1 COMPLETE — Export Folder Contents
  Location: /content/drive/MyDrive/Quantum_LPR_Export

📁 dataset/
  📄 2_train_hr_images.csv  (9.14 MB)
  📄 wYe7pBJ7-train.zip  (719.72 MB)
📁 checkpoints/
  📁 quantum/
    📄 best.pth  (2.83 MB)
    📄 history.json  (315 B)
    📄 latest.pth  (2.83 MB)
  📁 classical/
    📄 history.json  (64 B)
📁 meta/
  📄 test_indices.json  (0.10 MB)

  Total size: 734.6 MB

  Critical file check:
  ✅  dataset/wYe7pBJ7-train.zip  (719.72 MB)
  ✅  dataset/2_train_hr_images.csv  (9.14 MB)
  ✅  checkpoints/quantum/latest.pth  (2.83 MB)
  ✅  checkpoints/quantum/history.json  (0.00 MB)
  ✅  checkpoints/classical/history.json  (0.00 MB)

  🎉 Phase 1 complete!
  👉 Open your Drive and browse to: MyDrive/Quantum_LPR_Export/
  👉 Verify the files look correct
  👉 Then run PHASE 2 cells to push to HuggingFace


In [9]:
import pandas as pd

csv_path = os.path.join(EXPORT_DATASET, '2_train_hr_images.csv')

# ── Read with HEADER and correct separator ────────────────
df = pd.read_csv(csv_path, sep=',', header=0)

print("Columns:", list(df.columns))
print(f"Rows   : {len(df)}")
print("\nBEFORE:", df['path'].iloc[0])

# ── Strip the /content/lpr_train/ prefix ─────────────────
df['path'] = df['path'].str.replace(r'^.*?(?=train/)', '', regex=True)

print("AFTER :", df['path'].iloc[0])

# ── Verify ───────────────────────────────────────────────
still_abs = df['path'].str.startswith('/').sum()
print(f"\nStill absolute: {still_abs}  ← must be 0")

sample_ok = df['path'].iloc[0].startswith('train/')
print(f"Starts with 'train/': {sample_ok}  ← must be True")

# ── Save ────────────────────────────────────────────────
df.to_csv(csv_path, sep=',', index=False)
print(f"\n✅ Fixed CSV saved → {csv_path}")
print("   Row 0 path:", df['path'].iloc[0])
print("   Row 1 path:", df['path'].iloc[1])


Columns: ['path', 'label', 'track_id', 'type']
Rows   : 100000

BEFORE: train/train/Scenario-A/Brazilian/track_00975/hr-001.png
AFTER : train/Scenario-A/Brazilian/track_00975/hr-001.png

Still absolute: 0  ← must be 0
Starts with 'train/': True  ← must be True

✅ Fixed CSV saved → /content/drive/MyDrive/Quantum_LPR_Export/dataset/2_train_hr_images.csv
   Row 0 path: train/Scenario-A/Brazilian/track_00975/hr-001.png
   Row 1 path: train/Scenario-A/Brazilian/track_00975/hr-002.png


---
# 🚀 PHASE 2 — Push Export Folder → HuggingFace Hub
### ⚠️ Only run this after you have verified the Drive folder above!
---

## Phase 2 — Cell A: Authenticate with HuggingFace

In [10]:
!pip install huggingface_hub -q

from huggingface_hub import HfApi, login, whoami, create_repo

HF_TOKENS = [
    'ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS',
    'ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS',
]
HF_USERNAME     = 'Shanmuk4622'
HF_DATASET_REPO = f'{HF_USERNAME}/quantum-lpr-dataset'
HF_MODEL_REPO   = f'{HF_USERNAME}/quantum-lpr-checkpoints'

active_token = None
api = None
for tok in HF_TOKENS:
    try:
        login(token=tok, add_to_git_credential=False)
        info = whoami(token=tok)
        active_token = tok
        api = HfApi(token=tok)
        print(f'✅ Authenticated as: {info["name"]} (token: ...{tok[-8:]})')
        break
    except Exception as e:
        print(f'⚠️  Token ...{tok[-8:]} failed: {e}')

if not active_token:
    raise RuntimeError('All HF tokens failed.')

# Create repos if they don't exist
for repo_id, repo_type in [(HF_DATASET_REPO,'dataset'),(HF_MODEL_REPO,'model')]:
    try:
        create_repo(repo_id=repo_id, repo_type=repo_type, token=active_token, exist_ok=True)
        print(f'  ✅ Repo ready: {repo_id}')
    except Exception as e:
        print(f'  ❌ {repo_id}: {e}')

✅ Authenticated as: Shanmuk4622 (token: ...PrJjOlYJ)
  ✅ Repo ready: Shanmuk4622/quantum-lpr-dataset
  ✅ Repo ready: Shanmuk4622/quantum-lpr-checkpoints


## Phase 2 — Cell B: Upload CSV + Meta (small files, ~seconds)

In [11]:
import time

def push_file(local_path, hf_path, repo_id, repo_type, label):
    if not os.path.exists(local_path):
        print(f'  ⚠️  Skipping {label} — file not found locally')
        return
    size = os.path.getsize(local_path) / 1e6
    print(f'  📤 Uploading {label} ({size:.2f} MB)...')
    t0 = time.time()
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=hf_path,
        repo_id=repo_id,
        repo_type=repo_type,
        token=active_token,
        commit_message=f'Upload {label}'
    )
    print(f'  ✅ Done in {time.time()-t0:.1f}s → {repo_id}/{hf_path}')

print('🚀 Uploading small files (CSV, JSON)...')
print('─'*50)

# CSV to dataset repo
push_file(
    os.path.join(EXPORT_DATASET, '2_train_hr_images.csv'),
    'data/2_train_hr_images.csv',
    HF_DATASET_REPO, 'dataset', 'CSV Manifest'
)

# test_indices to both repos
idx_local = os.path.join(EXPORT_META, 'test_indices.json')
if os.path.exists(idx_local):
    push_file(idx_local, 'meta/test_indices.json', HF_DATASET_REPO, 'dataset', 'test_indices (dataset)')
    push_file(idx_local, 'meta/test_indices.json', HF_MODEL_REPO,   'model',   'test_indices (model)')

# Quantum history
push_file(
    os.path.join(EXPORT_Q_CKPT, 'history.json'),
    'quantum/history.json',
    HF_MODEL_REPO, 'model', 'Quantum history'
)

# Classical history
push_file(
    os.path.join(EXPORT_C_CKPT, 'history.json'),
    'classical/history.json',
    HF_MODEL_REPO, 'model', 'Classical history'
)

print('\n✅ Small files done.')

🚀 Uploading small files (CSV, JSON)...
──────────────────────────────────────────────────
  📤 Uploading CSV Manifest (7.24 MB)...


No files have been modified since last commit. Skipping to prevent empty commit.


  ✅ Done in 1.6s → Shanmuk4622/quantum-lpr-dataset/data/2_train_hr_images.csv
  📤 Uploading test_indices (dataset) (0.10 MB)...
  ✅ Done in 0.1s → Shanmuk4622/quantum-lpr-dataset/meta/test_indices.json
  📤 Uploading test_indices (model) (0.10 MB)...


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  ✅ Done in 0.1s → Shanmuk4622/quantum-lpr-checkpoints/meta/test_indices.json
  📤 Uploading Quantum history (0.00 MB)...
  ✅ Done in 0.1s → Shanmuk4622/quantum-lpr-checkpoints/quantum/history.json
  📤 Uploading Classical history (0.00 MB)...
  ✅ Done in 0.4s → Shanmuk4622/quantum-lpr-checkpoints/classical/history.json

✅ Small files done.


## Phase 2 — Cell C: Upload Quantum Checkpoint (.pth files)

In [12]:
print('🚀 Uploading Quantum checkpoints (.pth files)...')
print('─'*50)

push_file(
    os.path.join(EXPORT_Q_CKPT, 'latest.pth'),
    'quantum/latest.pth',
    HF_MODEL_REPO, 'model', 'Quantum latest.pth'
)

push_file(
    os.path.join(EXPORT_Q_CKPT, 'best.pth'),
    'quantum/best.pth',
    HF_MODEL_REPO, 'model', 'Quantum best.pth'
)

# Classical checkpoints (if they exist)
c_latest = os.path.join(EXPORT_C_CKPT, 'latest.pth')
c_best   = os.path.join(EXPORT_C_CKPT, 'best.pth')
if os.path.exists(c_latest):
    push_file(c_latest, 'classical/latest.pth', HF_MODEL_REPO, 'model', 'Classical latest.pth')
if os.path.exists(c_best):
    push_file(c_best,   'classical/best.pth',   HF_MODEL_REPO, 'model', 'Classical best.pth')

print('\n✅ Checkpoint uploads done.')

🚀 Uploading Quantum checkpoints (.pth files)...
──────────────────────────────────────────────────
  📤 Uploading Quantum latest.pth (2.83 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...points/quantum/latest.pth: 100%|##########| 2.83MB / 2.83MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoints/quantum/best.pth: 100%|##########| 2.83MB / 2.83MB            

No files have been modified since last commit. Skipping to prevent empty commit.


  ✅ Done in 0.7s → Shanmuk4622/quantum-lpr-checkpoints/quantum/best.pth

✅ Checkpoint uploads done.


## Phase 2 — Cell D: Upload Dataset ZIP (large file — may take 10–40 min)
> If this gets interrupted, just re-run this cell — `upload_large_folder` is **resumable**.

In [14]:
import time, os

zip_local = os.path.join(EXPORT_DATASET, 'wYe7pBJ7-train.zip')
zip_size  = os.path.getsize(zip_local) / 1e9

print(f'📦 Dataset ZIP: {zip_size:.2f} GB')
print('📤 Uploading to HuggingFace (resumable if interrupted)...')
print()

STAGING = '/content/hf_zip_stage'
os.makedirs(f'{STAGING}/data', exist_ok=True)

staged_zip = f'{STAGING}/data/wYe7pBJ7-train.zip'
if not os.path.exists(staged_zip):
    os.symlink(zip_local, staged_zip)
    print('  📂 Symlinked ZIP to staging dir.')
else:
    print('  📂 Already staged.')

t0 = time.time()
try:
    # ✅ FIX: no token= argument — relies on login() done earlier
    api.upload_large_folder(
        folder_path=STAGING,
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
        num_workers=4,
    )
    elapsed = (time.time() - t0) / 60
    print(f'\n✅ Dataset ZIP uploaded in {elapsed:.1f} minutes!')
except Exception as e:
    print(f'\n❌ Still failing: {e}')
    # Fallback: use regular upload_file if large_folder fails
    print('\n🔄 Trying fallback with upload_file...')
    t1 = time.time()
    api.upload_file(
        path_or_fileobj=zip_local,
        path_in_repo='data/wYe7pBJ7-train.zip',
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
        commit_message='Upload training dataset ZIP',
    )
    elapsed = (time.time() - t1) / 60
    print(f'✅ Uploaded via upload_file in {elapsed:.1f} minutes!')


📦 Dataset ZIP: 0.72 GB
📤 Uploading to HuggingFace (resumable if interrupted)...

  📂 Already staged.


Recovering from metadata files:   0%|          | 0/1 [00:00<?, ?it/s]




---------- 2026-04-12 09:10:17 (0:00:00) ----------
Files:   hashed 0/1 (0.0/719.7M) | pre-uploaded: 0/0 (0.0/719.7M) (+1 unsure) | committed: 0/1 (0.0/719.7M) | ignored: 0
Workers: hashing: 1 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 3
---------------------------------------------------


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e/data/wYe7pBJ7-train.zip:   1%|1         | 7.94MB /  720MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e/data/wYe7pBJ7-train.zip:   9%|8         | 63.9MB /  720MB            


✅ Dataset ZIP uploaded in 0.3 minutes!


## Phase 2 — Cell E: ✅ Final Verification on HuggingFace

In [15]:
print('🔍 Verifying HuggingFace repos...')
print()

all_good = True
for repo_id, repo_type in [(HF_DATASET_REPO,'dataset'),(HF_MODEL_REPO,'model')]:
    print(f'  📁 {repo_id}  [{repo_type}]')
    try:
        files = sorted(api.list_repo_files(repo_id=repo_id, repo_type=repo_type, token=active_token))
        for f in files:
            if f.startswith('.'): continue
            print(f'     ✅ {f}')
        if not [f for f in files if not f.startswith('.')]:
            print('     ⚠️  Repo is empty!')
            all_good = False
    except Exception as e:
        print(f'     ❌ Error listing files: {e}')
        all_good = False
    print()

print('='*60)
if all_good:
    print('🎉 SETUP COMPLETE! Both repos are ready.')
    print()
    print('Next steps:')
    print('  1. On Kaggle → Add-ons → Secrets → Add:')
    print('       HF_TOKEN_1 = ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS')
    print('       HF_TOKEN_2 = ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS')
    print('  2. Upload 01_Complete_Training_Kaggle.ipynb to Kaggle')
    print('  3. Set accelerator: GPU T4 x2')
    print('  4. Run! It will auto-resume from the quantum checkpoint.')
else:
    print('⚠️  Some issues found — check above and re-run the failed cells.')

🔍 Verifying HuggingFace repos...

  📁 Shanmuk4622/quantum-lpr-dataset  [dataset]
     ✅ README.md
     ✅ data/2_train_hr_images.csv
     ✅ data/wYe7pBJ7-train.zip
     ✅ meta/test_indices.json

  📁 Shanmuk4622/quantum-lpr-checkpoints  [model]
     ✅ README.md
     ✅ classical/history.json
     ✅ meta/test_indices.json
     ✅ quantum/best.pth
     ✅ quantum/history.json
     ✅ quantum/latest.pth

🎉 SETUP COMPLETE! Both repos are ready.

Next steps:
  1. On Kaggle → Add-ons → Secrets → Add:
       HF_TOKEN_1 = ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS
       HF_TOKEN_2 = ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS
  2. Upload 01_Complete_Training_Kaggle.ipynb to Kaggle
  3. Set accelerator: GPU T4 x2
  4. Run! It will auto-resume from the quantum checkpoint.
